# OCEAVERA — Biological Data Collection

**Branch work notebook**

**Branch:** `pipeline/biological-data-collection`  
**Review date:** 8 October 2026


## Contributors

Verified project identities for this branch:

<table style="width:100%; border-collapse:separate; border-spacing:1rem .2rem; margin:.45rem 0 1.25rem;">
<thead>
<tr>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Registration Number</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Member Name</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">GitHub Account</th>
</tr>
</thead>
<tbody>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">IT24100788</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Sanuda Abeysinghe</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><code>sanudaabey</code></td>
</tr>
</tbody>
</table>

This notebook describes branch-level work. It is not an individual contribution statement or assessed reflection.

## 1. Purpose and scope

This branch establishes an unrestricted Ocean Biodiversity Information System (OBIS) Area 230 collection workflow and a byte-preserving handoff for later source validation.

**Geographic context:** Area 230 is the publisher-defined collection scope. It is not asserted to equal the Sri Lankan EEZ or the eventual modelling domain.

### Included in this branch

- **Occurrence acquisition:** retrieve the complete Area 230 response set, including records marked as absences or dropped.
- **Format conversion:** preserve response fields while converting the verified JSON run to the access-page CSV schema.
- **Raw-data handoff:** validate and stage a selected raw CSV without changing its bytes.
- **Notebook demonstration:** audit the maintained scripts and check a CSV round-trip in memory.

### Outside this branch

- Scientific cleaning or record exclusion
- Focal-species selection or target construction
- Bio-ORACLE acquisition or integration
- Environmental feature preparation or modelling


## 2. Area 230 data at a glance

The complete Area 230 API run recorded the following:

<table style="width:100%; border-collapse:separate; border-spacing:1rem .2rem; margin:.45rem 0 1.25rem;">
<thead>
<tr>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Measure</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Recorded value</th>
<th scope="col" style="font-size:1.08em; font-weight:700; text-align:left; padding:.7rem 1rem; border-bottom:2px solid #64748b;">Meaning</th>
</tr>
</thead>
<tbody>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">API occurrence records</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><strong>23,934</strong></td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">All returned records are retained</td>
</tr>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">JSON response pages</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><strong>24</strong></td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Paginated source responses</td>
</tr>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">OBIS Area-page count</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><strong>23,327</strong></td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Publisher-page total, shown separately from the API run count.</td>
</tr>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Flagged records retained</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><strong>607</strong></td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Marked as absences, dropped records, or both; all are retained.</td>
</tr>
<tr>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">CSV fields</td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;"><strong>230 columns</strong></td>
<td style="font-size:.93em; text-align:left; vertical-align:top; padding:.7rem 1rem; border-bottom:1px solid #dbe2ea;">Returned API fields plus the documented access-page schema</td>
</tr>
</tbody>
</table>

The [source record](../../docs/records/2026-10-08-source-obis-area-230-api-csv.md) documents the counts, provenance and validation evidence.

### Input used by this notebook

1. Find the newest complete local Area 230 JSON run dynamically.
2. Verify its manifest, page checksums, cursor sequence, record count and unique IDs.
3. If no run directory is present, use a clearly labelled two-row synthetic fixture. It is a demonstration only, not OBIS evidence.

The local source is read-only. This notebook does not request new data from OBIS.

## 3. Branch workflow

1. **Retrieve all Area 230 API pages.** The [OBIS downloader](../../src/data_collection/obis_occurrences.py) saves the unchanged paginated JSON responses.
2. **Convert the verified JSON run.** The [JSON-to-CSV converter](../../src/data_collection/obis_json_to_csv.py) validates the run and writes the documented CSV schema.
3. **Stage the selected raw CSV.** The [CSV staging utility](../../src/data_preparation/stage_obis_csv.py) validates the input and copies it byte for byte to the stable interim phase.
4. **Review the workflow.** This notebook checks script syntax and demonstrates CSV conversion and round-trip verification in memory.

**Implementation:** The maintained Python scripts carry out these steps. This notebook reviews them without duplicating their implementations.


## 4. Execution and data handling

### Execution order

1. Read the opening scope and source-data context.
2. Execute the Python code cells in order.
3. Review the aggregate verification summary; no occurrence-level preview is stored.

### Data safeguards

- **Network:** the notebook runs offline by default and makes no API requests.
- **Input:** the newest complete local JSON run is read-only.
- **Conversion:** CSV rows are held in an in-memory text stream.
- **Persistence:** no dataset file or converted CSV is written to disk.
- **Saved output:** aggregate verification only; occurrence rows and coordinates are omitted.

The checks verify source integrity and CSV serialization. They do not establish whether the observations are scientifically suitable for modelling.


## 5. Review the maintained scripts

### 5.1 Locate the scripts and check syntax

The following cells locate the repository and parse the three maintained intake scripts. They do not invoke command-line entry points or begin data acquisition.

**Scripts reviewed:** the OBIS downloader, the JSON-to-CSV converter and the raw-to-interim staging utility.


In [9]:
import ast
import csv
import io
import json
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
ROOT = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "src" / "data_collection" / "obis_occurrences.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the OCEAVERA repository or one of its subfolders.")

def repo_relative_path(path):
    return Path(path).resolve().relative_to(ROOT).as_posix()

print("Project root located.")
print("Python runtime:", sys.version.split()[0])
print("Notebook example uses Python's standard library only.")


Project root located.
Python runtime: 3.14.8
Notebook example uses Python's standard library only.


In [10]:
source_paths = {
    "OBIS downloader": ROOT / "src" / "data_collection" / "obis_occurrences.py",
    "JSON-to-CSV converter": ROOT / "src" / "data_collection" / "obis_json_to_csv.py",
    "CSV staging utility": ROOT / "src" / "data_preparation" / "stage_obis_csv.py",
}

source_audit = {}
for label, path in source_paths.items():
    source_text = path.read_text(encoding="utf-8")
    syntax_tree = ast.parse(source_text, filename=path.name)
    compile(syntax_tree, repo_relative_path(path), "exec")
    source_audit[label] = {
        "lines": len(source_text.splitlines()),
        "functions": sum(
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            for node in syntax_tree.body
        ),
    }

for label, details in source_audit.items():
    print(f"PASS | {label}: valid Python, {details['lines']:,} lines")


PASS | OBIS downloader: valid Python, 545 lines
PASS | JSON-to-CSV converter: valid Python, 500 lines
PASS | CSV staging utility: valid Python, 329 lines


### 5.2 Load the conversion helpers

The converter's read-only helper functions are loaded into notebook memory. The following code cell does not invoke the converter command-line interface or create a CSV file.


In [11]:
converter_path = source_paths["JSON-to-CSV converter"]
converter_scope = {
    "__name__": "oceavera_obis_json_to_csv_notebook",
    "__file__": str(converter_path),
}
converter_source = converter_path.read_text(encoding="utf-8")
exec(compile(converter_source, repo_relative_path(converter_path), "exec"), converter_scope)
print("PASS | Read-only JSON validation and CSV serialization helpers loaded.")


PASS | Read-only JSON validation and CSV serialization helpers loaded.


## 6. Select and verify the source records

The notebook selects the newest complete local Area 230 run. Before conversion, it verifies:

- Manifest query and completion status
- Page file hashes and page record counts
- Pagination cursor order and page ID boundaries
- Total record count and unique occurrence IDs
- The documented CSV columns

If no complete local run exists, a two-row synthetic fixture keeps the demonstration executable. Its outputs are clearly labelled; the fixture is not OBIS evidence.


In [12]:
JSON_ROOT = ROOT / "data" / "raw" / "obis" / "json"
fixture_records = (
    {
        "id": "fixture-1",
        "scientificName": "Synthetic example taxon A",
        "date_year": 2024,
        "decimalLatitude": 6.0,
        "decimalLongitude": 81.0,
    },
    {
        "id": "fixture-2",
        "scientificName": "Synthetic example taxon B",
        "date_year": 2025,
        "decimalLatitude": 7.0,
        "decimalLongitude": 82.0,
    },
)
candidate_runs = sorted(
    (item for item in JSON_ROOT.glob("area-230-*") if item.is_dir()),
    reverse=True,
)
run_dir = None
manifest = None
fields_not_returned = []

print("Source run folders discovered:", len(candidate_runs))


Source run folders discovered: 1


In [13]:
if candidate_runs:
    run_dir, manifest = converter_scope["resolve_run"](None)
    columns, record_count, fields_not_returned = converter_scope["check_manifest"](
        run_dir, manifest
    )
    source_label = "Verified saved Area 230 JSON"
    print(f"PASS | Source run: {repo_relative_path(run_dir)}")
    print(f"PASS | Manifest records: {record_count:,}; pages: {len(manifest['pages']):,}")
    print(f"PASS | Unique IDs: {manifest['unique_record_ids']:,}; CSV columns: {len(columns):,}")
    print(f"Schema columns absent from these API responses: {len(fields_not_returned):,} (retained as blank CSV columns)")
else:
    columns = sorted(
        set(converter_scope["DATA_ACCESS_FIELDS"]).union(
            *(record.keys() for record in fixture_records)
        )
    )
    record_count = len(fixture_records)
    source_label = "Offline synthetic fixture"
    print("No local Area 230 run found; using a labelled synthetic fixture.")
    print(f"Fixture rows: {record_count}; schema columns: {len(columns):,}")


PASS | Source run: data/raw/obis/json/area-230-20261008T055217394846Z
PASS | Manifest records: 23,934; pages: 24
PASS | Unique IDs: 23,934; CSV columns: 230
Schema columns absent from these API responses: 12 (retained as blank CSV columns)


## 7. Build the CSV representation in memory

The following code cell serializes records to an in-memory `io.StringIO` stream using the maintained converter's field mapping and serialization rules. Structured JSON values and the documented field alias are preserved.

**Output:** the CSV text remains in memory for round-trip verification; no file is written.


In [14]:
def csv_row(record):
    return {
        field: converter_scope["csv_cell"](
            converter_scope["csv_value"](record, field)
        )
        for field in columns
    }

csv_buffer = io.StringIO(newline="")
writer = csv.DictWriter(
    csv_buffer,
    fieldnames=columns,
    extrasaction="raise",
    lineterminator="\n",
)
writer.writeheader()

source_records = (
    converter_scope["iter_records"](run_dir, manifest)
    if run_dir is not None
    else iter(fixture_records)
)
written_rows = 0
for record in source_records:
    writer.writerow(csv_row(record))
    written_rows += 1

if written_rows != record_count:
    raise AssertionError("The in-memory CSV row count differs from the verified source count.")

csv_buffer.seek(0)
csv_size_bytes = len(csv_buffer.getvalue().encode("utf-8"))
print(f"Prepared {written_rows:,} rows and {len(columns):,} columns in memory.")
print(f"In-memory CSV size: {csv_size_bytes:,} bytes; no dataset file was written.")


Prepared 23,934 rows and 230 columns in memory.
In-memory CSV size: 26,892,051 bytes; no dataset file was written.


## 8. Verify the round-trip

Each parsed CSV row is compared with its source record after the converter's documented serialization. This detects extra or missing rows and field-level differences.

- **Rows:** every source record is checked.
- **Columns:** every column is compared for every row.
- **Persistence:** no CSV file is saved.


In [15]:
csv_buffer.seek(0)
reader = csv.DictReader(csv_buffer)
source_records = (
    converter_scope["iter_records"](run_dir, manifest)
    if run_dir is not None
    else iter(fixture_records)
)
sentinel = object()
checked_rows = 0

for actual_row in reader:
    source_record = next(source_records, sentinel)
    if source_record is sentinel:
        raise AssertionError("The CSV contains more rows than the source records.")
    if actual_row != csv_row(source_record):
        raise AssertionError(f"CSV content differs from the source at row {checked_rows + 1:,}.")
    checked_rows += 1

if next(source_records, sentinel) is not sentinel:
    raise AssertionError("The CSV contains fewer rows than the source records.")
if checked_rows != record_count:
    raise AssertionError("The round-trip count differs from the verified record count.")

print(f"PASS | {checked_rows:,} rows matched their source values after CSV parsing.")
print(f"PASS | All {len(columns):,} columns were checked for every row.")


PASS | 23,934 rows matched their source values after CSV parsing.
PASS | All 230 columns were checked for every row.


## 9. Results summary

The summary identifies whether the input was a verified local collection or the synthetic fallback. Saved output contains aggregate verification details only; occurrence identifiers, taxa, dates and coordinates are omitted.


In [16]:
print("OCEAVERA | Biological data collection verification")
print("Input:", source_label)
if run_dir is not None:
    print("Verified run:", repo_relative_path(run_dir))
print(f"Records checked: {checked_rows:,}")
print(f"CSV columns: {len(columns):,}")
print("Round-trip: PASS")
print("Persistence: in-memory only")
print()

print("Record-level output: omitted.")


OCEAVERA | Biological data collection verification
Input: Verified saved Area 230 JSON
Verified run: data/raw/obis/json/area-230-20261008T055217394846Z
Records checked: 23,934
CSV columns: 230
Round-trip: PASS
Persistence: in-memory only

Record-level output: omitted.


## 10. What the checks establish

### Verified in this run

- The three maintained scripts parse as valid Python without running the downloader.
- The saved Area 230 run is discovered dynamically and checked against its manifest.
- All source records are serialized to an in-memory CSV stream.
- Every parsed row and all declared columns pass the round-trip comparison.
- No network request or dataset-file write occurs.

### Limits

A manifest-complete run is verified against its own recorded totals and hashes. This does not independently establish that OBIS has published no newer records. The notebook does not perform scientific cleaning, establish absence, select a species, integrate environmental data or evaluate a model.


## 11. References

### Collection scope and provenance

- [D-034 — OBIS Area 230 collection scope](../../docs/records/2026-10-07-decision-obis-area-230.md)
- [D-036 — OBIS API JSON and CSV method](../../docs/records/2026-10-08-decision-obis-area-230-api-csv.md)
- [Area 230 source record](../../docs/records/2026-10-08-source-obis-area-230-api-csv.md)

### Repository conventions

- [D-038 — phase-specific derived-data paths](../../docs/records/2026-10-08-decision-phase-specific-derived-data-paths.md)
- [D-039 — Python 3.14 baseline](../../docs/records/2026-10-08-decision-python-3-14-baseline.md)
- [D-040 — CI and intake syntax checks](../../docs/records/2026-10-08-decision-ci-branch-and-intake-syntax-checks.md)
- [D-041 — branch work notebook purpose and structure](../../docs/records/2026-10-08-decision-branch-work-notebooks.md)
- [D-042 — contributor table and metadata](../../docs/records/2026-10-08-decision-branch-notebook-member-table.md)
- [D-043 — environment and package approval](../../docs/records/2026-10-08-decision-python-environment-approval.md)
- [D-044 — notebook execution and in-memory data handling](../../docs/records/2026-10-08-decision-executable-branch-notebooks.md)
- [D-045 — native Markdown and executable Code cells](../../docs/records/2026-10-08-decision-branch-notebook-cell-types.md)

### Project guidance

- [Notebook convention](../README.md#branch-work-notebook-convention)
- [Project requirements](../../PROJECT_REQUIREMENTS.md)
- [Readiness review](../../docs/project/repository-readiness-and-alignment.md)
- [Data provenance guide](../../docs/data/data-storage-and-provenance.md)
